# Exercise 1 · Keep only the tests that can fail

Codex wrote five tests, and all five pass. You break the code on purpose to see which tests notice.

**Goal:** Break the code on purpose, and keep only the generated tests that notice.

**Why it matters:** Generated tests often pass because they agree with the code. A test counts only after you see it fail on wrong code.

**Before you start:** You installed Codex and signed in, and you can leave it with `/exit` (Monday's exercises).

**What the check measures:** `test_cases.py` keeps 2 tests or more. Each passes on `cases.py` and fails on `broken/cases.py`. Your answer to one question is right.

**Done when:** the last cell prints `Done:`.

## The situation

`cases.py` holds `filter_cases`, which picks the cases with one status. Codex wrote `test_cases.py` for it. `broken/cases.py` is a copy with a fault: it returns every case. The files hold made-up data.

## Step 1 · Write the practice files

Run the cell. It writes four files into `practice/`, beside this notebook. It never overwrites a file.

Expect four lines that start with `Wrote practice/`, or `Kept` for a file that is already there.

In [ ]:
FILES = {
    "cases.py": """def filter_cases(cases: list[dict], status: str) -> list[dict]:
    \"\"\"Return the cases whose status is status, in their order.

    Args:
        cases: Dicts, each with a "status" key.
        status: The status to keep.

    Returns:
        The matching cases.
    \"\"\"
    return [case for case in cases if case["status"] == status]
""",
    "test_cases.py": """# Tests Codex wrote (simulated).
from cases import filter_cases

CASES = [
    {"id": "C-1", "status": "open"},
    {"id": "C-2", "status": "open"},
    {"id": "C-3", "status": "closed"},
]


def test_filter_runs() -> None:
    filter_cases(CASES, "open")


def test_filter_returns_cases() -> None:
    assert len(filter_cases(CASES, "open")) >= 0


def test_filter_is_consistent() -> None:
    assert filter_cases(CASES, "open") == filter_cases(CASES, "open")


def test_open_returns_only_open_cases() -> None:
    assert [case["status"] for case in filter_cases(CASES, "open")] == ["open", "open"]


def test_unknown_status_returns_no_cases() -> None:
    assert filter_cases(CASES, "pending") == []
""",
    "broken/cases.py": """# A broken copy, for Step 3.


def filter_cases(cases: list[dict], status: str) -> list[dict]:
    \"\"\"Return the cases whose status is status, in their order.

    Args:
        cases: Dicts, each with a "status" key.
        status: The status to keep.

    Returns:
        The matching cases.
    \"\"\"
    return list(cases)
""",
    ".gitignore": """__pycache__/
""",
}

from pathlib import Path

PRACTICE = Path("practice")
for name, text in FILES.items():
    path = PRACTICE / name
    if path.exists():
        print(f"Kept practice/{name}: it is already there.")
    else:
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(text, encoding="utf-8")
        print(f"Wrote practice/{name}")

## Step 2 · Open a terminal and commit the start

Choose **Terminal > New Terminal**. Type each line, then press Enter:

```sh
source .venv/bin/activate
cd week-02/day-03/exercise-01-tests-that-can-fail/practice
git init -q -b main && git add -A && git commit -qm "Start"
```

Expect no output from the last line.

## Step 3 · Run the tests, then break the code on purpose

pytest runs each function named `test_...` in each file named `test_*.py`. A test fails when its `assert` is false, or when the code raises an error.

```sh
python -m pytest -q
```

Expect `5 passed`. Now break the code. The first line puts the broken copy in place of `cases.py`. The second line runs the tests. The third line puts the real file back.

```sh
cp broken/cases.py cases.py
python -m pytest -v
git restore cases.py
```

`-v` prints one line for each test. Expect `PASSED` for the first three tests, `FAILED` for the last two, and then `2 failed, 3 passed`.

### Learn: a test that passes first time agrees with the code as it is

TestGen-LLM is Meta's tool that uses a language model to improve tests that people wrote. It keeps only the generated tests that pass: "TestGen-LLM discards any test case that does not pass on first execution" ([Alshahwan and others, 2024](https://arxiv.org/html/2402.09171)). Such a test agrees with the code as it is, faults included.

## Step 4 · Choose the tests to keep

Open `practice/test_cases.py`. Read the three tests that passed on the broken copy. What fault could each one catch?

<details><summary>Show the answer</summary>

Delete `test_filter_runs`, `test_filter_returns_cases` and `test_filter_is_consistent`. Why: none can fail when the result is wrong. The first has no `assert`. The second asserts a length of 0 or more, which is always true. The third compares the filter with itself. Keep the two tests that failed on the broken copy.

</details>

## Step 5 · Ask Codex to delete the tests that never fail

```sh
codex --no-daemon
```

If Codex stops with an error about `--no-daemon`, run `codex` without it. If Codex asks "Trust this folder?", choose `1. Trust and continue`. Type this request with your three names:

```text
Delete these tests from test_cases.py: <the three names>. Change no other file. Done when python -m pytest -q shows 2 passed.
```

Type `/diff` and read the change. Press `q`, then type `/exit`.

Expect a diff that deletes three tests and nothing else.

## Step 6 · Run both ways again, then commit

```sh
python -m pytest -q
cp broken/cases.py cases.py
python -m pytest -q
git restore cases.py
git commit -qam "Keep only the tests that can fail"
```

Expect `2 passed`, then `2 failed`.

## Step 7 · Answer one question

Codex writes twelve tests, and all twelve pass first time. What does that tell you?

- **A.** The code is correct.
- **B.** The tests agree with the code as it is, faults included.
- **C.** Each test would fail if the code were wrong.

Write your letter between the quotes, then run the cell.

<details><summary>Show the answer</summary>

B. Why: in Step 3, three passing tests also passed on code with a fault.

</details>

In [ ]:
ANSWER = ""
print(f"Saved your answer: {ANSWER}")

## Start over (only if you need it)

Set `START_OVER = True` and run the cell. It writes the Step 1 files again, with all five tests. Git keeps your commits. Then set it back to `False` and go to Step 3.

Expect `Nothing changed.` while it is `False`.

In [ ]:
# Start over: only if START_OVER is True, write the Step 1 files again
START_OVER = False

if not START_OVER:
    print("Nothing changed.")
elif "FILES" not in globals():
    print("Run the Step 1 cell first, then this cell.")
else:
    for name, text in FILES.items():
        (PRACTICE / name).parent.mkdir(parents=True, exist_ok=True)
        (PRACTICE / name).write_text(text, encoding="utf-8")
    print("Wrote the practice files again. Now set START_OVER back to False.")

## Check your work

Run the cell. It runs your tests on both copies of `cases.py`, in a temporary copy of `practice/`. Expect three `ok:` lines, then `Done:`. A `Not yet:` line names the first problem and what to do.

In [ ]:
# Check your work: run the tests on cases.py and on the broken copy
import shutil
import subprocess
import sys
import tempfile
import xml.etree.ElementTree as ET
from pathlib import Path

PRACTICE = Path("practice")


def failed_tests(broken):
    """Return {test name: True if it failed}, or None if pytest did not run."""
    with tempfile.TemporaryDirectory() as tmp:
        work = shutil.copytree(PRACTICE, Path(tmp) / "p", ignore=shutil.ignore_patterns(".git", "__pycache__"))
        if broken:
            shutil.copyfile(work / "broken" / "cases.py", work / "cases.py")
        subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "test_cases.py",
                        "--junitxml=r.xml"], cwd=work, capture_output=True, timeout=300)
        if not (work / "r.xml").is_file():
            return None
        return {case.get("name"): any(case.find(tag) is not None for tag in ("failure", "error", "skipped"))
                for case in ET.parse(work / "r.xml").iter("testcase")}


def first_problem():
    """Print ok for each check that passes; return the first problem, or ""."""
    good = failed_tests(broken=False)
    if good is None:
        return "pytest did not run. Run the Step 1 cell, then python -m pytest -q in practice/ (Step 3)."
    failing = [name for name, failed in good.items() if failed]
    if failing:
        return f"{failing[0]} fails on cases.py. Run python -m pytest -q and read why (Step 6)."
    if len(good) < 2:
        return f"test_cases.py has {len(good)} test. Keep the 2 tests that failed on the broken copy (Step 4)."
    print(f"ok: all {len(good)} tests pass on cases.py")
    bad = failed_tests(broken=True) or {}
    never = [name for name in good if not bad.get(name)]
    if never:
        return f"{never[0]} passes on broken/cases.py too, so it can never fail. Delete it (Step 5)."
    print("ok: each test fails on broken/cases.py")
    choice = str(globals().get("ANSWER", "")).strip().upper()
    if not choice:
        return "the question has no answer. Write a letter in the Step 7 cell and run it."
    if choice != "B":
        return f"answer {choice} is not right. Read the Learn cell after Step 3, then try again."
    print("ok: your answer is right")
    return ""


try:
    problem = first_problem()
except Exception as error:
    problem = f"the check stopped on {type(error).__name__}: {error}. Run the Step 1 cell, then this cell."
print(f"Not yet: {problem}" if problem else "Done: every test you kept passes on the code and fails on the broken copy.")

## What you learned

- A generated test counts only if it can fail. Run it on code you broke on purpose to see that.

**Reflect:** Which tests in your own work have you never seen fail?